<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### **Unit of Analysis + Time Window**

**Unit of Analysis:** One row = one content page (a specific URL/article)

**Time Window**:

- **Feature window:** 90 days of historical data (e.g., March-May 2026)

- **Target window:** The following 30 days (e.g., June 2026) to measure outcomes

- **Decision point:** The date when the feature window ends and the target window begins

**Why this matters:**

- This prevents leakage - features come from BEFORE the target is measured

- It matches the real-world decision: "Based on the past 90 days, what will happen in the next 30 days?"

In [2]:
# Verify the unit of analysis with the starter dataset

import os, sys, subprocess
import pandas as pd
import numpy as np

# Setup for Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

# Load starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== Unit of Analysis ===\n")
print("One row = one content page")
print(f"Total rows (pages): {len(df):,}")
print(f"Total clients: {df['client_id'].nunique():,}")
print(f"One row per content_id? {df['content_id'].nunique() == len(df)} \n")

# Show the time windows in the starter data
print("=== Time Windows ===\n")
print("The starter data uses trailing 90-day metrics:")
print(f"  impressions_90d = impressions over the last 90 days")
print(f"  trend_direction = computed from trend_pct (historical trajectory)")
print("\nThe warehouse release supports custom time windows.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
=== Unit of Analysis ===

One row = one content page
Total rows (pages): 30,000
Total clients: 32
One row per content_id? True 

=== Time Windows ===

The starter data uses trailing 90-day metrics:
  impressions_90d = impressions over the last 90 days
  trend_direction = computed from trend_pct (historical trajectory)

The warehouse release supports custom time windows.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: Feature / Label / Context / Excluded

### Features (Signals I'll use to explain/rank)

| Field | Why it's a feature |
|---|---|
| `avg_position` | Average search result position — a known signal of visibility |
| `ctr` | Click-through rate — measures how attractive the listing is |
| `engagement_rate` | Visitor engagement — measures content quality |
| `content_age_days` | Age of the content — freshness signal |
| `word_count` | Content length — structural signal |
| `search_volume` | Keyword demand — potential audience size |
| `competition` | How hard it is to rank — difficulty signal |

### Labels (What I'm trying to predict/explain)

| Field | Why it's a label |
|---|---|
| `impressions_90d` | Visibility measure — higher = more visible |
| `is_declining_label` | Performance trajectory — declining = needs attention |

### Context (Used for grouping, not as features)

| Field | How I'll use it |
|---|---|
| `content_id` | Grouping, never as a feature |
| `client_id` | Grouped train/test splits, never as a feature |

### Excluded (Not used)

| Field | Why excluded |
|---|---|
| `trend_pct` | **LEAKAGE!** Derived from the label |
| `trend_direction` | **LEAKAGE!** Derived from the label |
| `competition_level` | Redundant (duplicates `competition` numeric) |
| Any tier columns | Redundant (duplicates numeric columns) |

> **CRITICAL: The Label Trap**
>
> `trend_direction` and `trend_pct` are NEVER features. They're how the label was created. Using them would leak the answer — the model would look perfect but teach us nothing.

In [3]:
# Show the fields in action

import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== Feature/Label/Context/Excluded Summary ===\n")

# Features
features = ['avg_position', 'ctr', 'engagement_rate', 'content_age_days',
            'word_count', 'search_volume', 'competition']

print("Features (7):")
for f in features:
    non_null = df[f].notna().sum()
    print(f"  - {f}: {non_null:,} non-null rows")
print()

# Labels
labels = ['impressions_90d', 'trend_direction']
print("Labels (2):")
for l in labels:
    non_null = df[l].notna().sum()
    print(f"  - {l}: {non_null:,} non-null rows")
print()

# Context
context = ['content_id', 'client_id']
print("Context (2):")
for c in context:
    unique = df[c].nunique()
    print(f"  - {c}: {unique:,} unique values")
print()

# Excluded
excluded = ['trend_pct', 'competition_level']
print("Excluded (leakage/duplicates):")
for e in excluded:
    if e in df.columns:
        print(f"  - {e}: EXCLUDED (would leak the answer or duplicate another column)")
print()

print("\n All features are knowable before the target window.")
print(" trend_pct and trend_direction are EXCLUDED to prevent leakage.")

=== Feature/Label/Context/Excluded Summary ===

Features (7):
  - avg_position: 30,000 non-null rows
  - ctr: 30,000 non-null rows
  - engagement_rate: 30,000 non-null rows
  - content_age_days: 30,000 non-null rows
  - word_count: 22,301 non-null rows
  - search_volume: 27,532 non-null rows
  - competition: 27,532 non-null rows

Labels (2):
  - impressions_90d: 30,000 non-null rows
  - trend_direction: 30,000 non-null rows

Context (2):
  - content_id: 30,000 unique values
  - client_id: 32 unique values

Excluded (leakage/duplicates):
  - trend_pct: EXCLUDED (would leak the answer or duplicate another column)
  - competition_level: EXCLUDED (would leak the answer or duplicate another column)


 All features are knowable before the target window.
 trend_pct and trend_direction are EXCLUDED to prevent leakage.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### **Verification Queries**
Every claim above is verified with a query below. A contract claim without a query next to it is a guess.

In [4]:
# Query 1: Grain Verification
print("=== Query 1: Grain Verification ===\n")
print("Purpose: Confirm one row = one content page")
print("\nSQL Equivalent (on dim_content):")
print("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT content_hash_id) AS unique_content,
  COUNT(*) = COUNT(DISTINCT content_hash_id) AS one_row_per_content
FROM dim_content
""")

# Using starter data as proxy
print("\nResult on starter data (proxy for warehouse):")
print(f"Total rows: {len(df):,}")
print(f"Unique content_ids: {df['content_id'].nunique():,}")
print(f"One row per content_id? {df['content_id'].nunique() == len(df)} ✅")
print()

# Query 2: Row Count and Date Span
print("=== Query 2: Row Count and Date Span ===\n")
print("Purpose: Show the slice has enough data")
print("\nSQL Equivalent (on fact_content_daily_performance_sample):")
print("""
SELECT
  MIN(report_date) AS earliest_date,
  MAX(report_date) AS latest_date,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT content_hash_id) AS unique_pages,
  COUNT(DISTINCT client_hash_id) AS unique_clients
FROM fact_content_daily_performance_sample
""")

print("\nResult on starter data (proxy):")
print(f"Total rows: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print(f"Date range: 90-day trailing window")

# Query 3: Availability Check
print("\n=== Query 3: Availability Check ===\n")
print("Purpose: Show which rows have usable data")
print("\nSQL Equivalent (with IS TRUE):")
print("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE impressions IS NOT NULL AND impressions > 0) AS rows_with_data,
  ROUND(COUNT(*) FILTER (WHERE impressions IS NOT NULL AND impressions > 0) * 100.0 / COUNT(*), 2) AS pct_with_data
FROM fact_content_daily_performance_sample
WHERE ga4_data_available = TRUE
""")

# Availability check on starter data
has_impressions = df['impressions_90d'] > 0
has_sessions = df['sessions_90d'] > 0
has_ctr = df['ctr'] > 0

print("\nResult on starter data (proxy):")
print(f"Total rows: {len(df):,}")
print(f"Rows with impressions > 0: {has_impressions.sum():,} ({has_impressions.mean()*100:.1f}%)")
print(f"Rows with sessions > 0: {has_sessions.sum():,} ({has_sessions.mean()*100:.1f}%)")
print(f"Rows with CTR > 0: {has_ctr.sum():,} ({has_ctr.mean()*100:.1f}%)")

=== Query 1: Grain Verification ===

Purpose: Confirm one row = one content page

SQL Equivalent (on dim_content):

SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT content_hash_id) AS unique_content,
  COUNT(*) = COUNT(DISTINCT content_hash_id) AS one_row_per_content
FROM dim_content


Result on starter data (proxy for warehouse):
Total rows: 30,000
Unique content_ids: 30,000
One row per content_id? True ✅

=== Query 2: Row Count and Date Span ===

Purpose: Show the slice has enough data

SQL Equivalent (on fact_content_daily_performance_sample):

SELECT 
  MIN(report_date) AS earliest_date,
  MAX(report_date) AS latest_date,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT content_hash_id) AS unique_pages,
  COUNT(DISTINCT client_hash_id) AS unique_clients
FROM fact_content_daily_performance_sample


Result on starter data (proxy):
Total rows: 30,000
Unique clients: 32
Date range: 90-day trailing window

=== Query 3: Availability Check ===

Purpose: Show which rows have usable data

SQL 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### **Data Limits**
**What this data can NEVER tell me:**

1. **Causation -** I can observe correlations but never prove that a signal causes performance

2. **Algorithm details -** I can't reverse-engineer Google's search algorithm from this data

3. **Why people don't click -** CTR tells me they didn't click, but not WHY (title? snippet? position? intent mismatch?)

4. **Content quality -** Word count and engagement rate are proxies, not direct measures of quality

5. **What happens after refresh -** This data alone can't prove that updating a page caused a recovery

**Limitations of this specific slice:**

- Visible pages only - I'll filter to pages with impressions ≥ 100 to avoid noise

- Historical constraints - Different clients have different history lengths (check `gsc_data_start`)

- GA4 data gaps - Before a client's `ga4_data_start`, GA4 columns are zero-filled with `ga4_data_available = FALSE`

- Sparse AI data - AI-referral sessions are extremely rare (30,177 rows out of 78.8M)

- Unbalanced panel - Not all clients have equal history; some have much more data than others

**What I'll do about it:**

- Use per-client time windows instead of one global window

- Filter `ga4_data_available` = TRUE when using GA4 metrics

- Require minimum volume (impressions ≥ 100) for reliable analysis

- Use careful language: "observed," "measured," "directional"

In [5]:
# Show the data limits

print("=== Data Limits ===\n")

# Show the sparsity of AI data
print("Sparsity Example (on starter data):")
ai_pct = df['ai_sessions_90d'].mean()
ai_rows = (df['ai_sessions_90d'] > 0).sum()
print(f"  Average AI sessions: {ai_pct:.3f}")
print(f"  Rows with any AI sessions: {ai_rows:,} ({ai_rows/len(df)*100:.2f}%)\n")

# Show the GA4/engagement gaps
print("GA4 Data Gaps:")
has_ga4 = (df['engagement_rate'] > 0) | (df['scroll_rate'] > 0)
print(f"  Rows with engagement data: {has_ga4.sum():,} ({has_ga4.mean()*100:.1f}%)\n")

print("What this means:")
print("  - AI data is very sparse — treat with caution")
print("  - Engagement data is missing for many rows")
print("  - I'll filter to pages with enough data for reliable analysis")

=== Data Limits ===

Sparsity Example (on starter data):
  Average AI sessions: 0.204
  Rows with any AI sessions: 1,930 (6.43%)

GA4 Data Gaps:
  Rows with engagement data: 18,709 (62.4%)

What this means:
  - AI data is very sparse — treat with caution
  - Engagement data is missing for many rows
  - I'll filter to pages with enough data for reliable analysis


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.